In [119]:
from langchain_community.document_loaders import TextLoader

In [120]:
loader = TextLoader('sample-text.txt')
data = loader.load()

In [121]:
data[0].page_content

'The process of making maple syrup begins by tapping a spout (sometimes called a spile) into the sugar maple tree. The spile is inserted into the tree about 2 inches deep and the sap is collected as it flows out. The sap is then taken to a sugar shack where it is boiled down to concentrate the sugars. As the sap boils, water in the sap is evaporated and the syrup becomes more and more thick. Once the syrup reaches the right sugar content, which is usually when the boiling point reaches 219 degrees Fahrenheit, it is bottled and enjoyed.'

In [122]:
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader('movies.csv', source_column='title')
data = loader.load()

In [123]:
data[1].metadata

{'source': 'Doctor Strange in the Multiverse of Madness', 'row': 1}

In [124]:
from langchain_community.document_loaders import UnstructuredURLLoader

In [133]:
loader = UnstructuredURLLoader(urls = [
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html",
])
data = loader.load()

In [134]:
data[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nEco Pulse\n\nMC Learn\n\nTrending Topics\n\nSensex Today\n\nLenskart Share Price\n\nPaytm Shares\n\nBuzzing Stocks\n\nOnEMI Tech Shares\n\nWall Street rises as Tesla soars o

In [127]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [135]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

docs = text_splitter.split_documents(data)
len(docs)

11

In [136]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nEco Pulse\n\nMC Learn\n\nTrending Topics\n\nSensex Today\n\nLenskart Share Price\n\nPaytm Shares\n\nBuzzing Stocks\n\nOnEMI Tech Shares\n\nWall Street rises as Tesla soars o

In [137]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import faiss

embeddings = OpenAIEmbeddings(api_key=api_key)
vector_index =  FAISS.from_documents(docs, embeddings)

In [142]:
vector_index.save_local("faiss_index")

In [139]:
import pickle
#file_path = "vector_index.pkl"
# with open(file_path, "wb") as f:
#     pickle.dump(vector_index, f)

In [157]:
file_path = "faiss_index"
vectorIndex = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
retriever = vectorIndex.as_retriever(
    search_kwargs={"k": 4},
     search_type="similarity"
)

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a7fff3ce8d0>, search_kwargs={'k': 4})

In [ ]:
import langchain_core

query = "what is the price of Tiago ICNG?"
docs = retriever.invoke(query)
# Display retrieved documents
# for i, doc in enumerate(docs, 1):
#     print(f"\n--- Document {i} ---")
#     print(doc.page_content)
#     print("Metadata:", doc.metadata)

In [183]:
from langchain_openai import OpenAI
llm = llm = OpenAI(api_key = api_key, temperature=0.5)
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_KEY")

In [ ]:
from langchain_core.prompts  import ChatPromptTemplate 
prompt = ChatPromptTemplate.from_template(
"""
Use the following portion of a long document ot see if any of the text is relevant to answer the question.
Return any relevant text verbatim.

Context:
{context}

Question:
{question}

Answer:
"""
)
from langchain_core.runnables import RunnablePassthrough 
chain = ({"context": retriever, "question": RunnablePassthrough()} |
            prompt|llm
        )
langchain_core.globals.set_debug(False) 
chain.invoke(query)

'\nThe price of Tiago ICNG starts at Rs. 7.1 lakh. Source: https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html'

In [223]:
langchain_core.globals.set_debug(True) 
map_results = []
for doc in docs:
    result = llm.invoke(
        prompt.format(
            context=doc.page_content,
            question=query
        )
    )
    map_results.append(result)
map_prompt = ChatPromptTemplate.from_template("""
"Given the following extracted parts of a long document and a question, create a final answer with references (\"SOURCES\"). \nIf you don't know the answer, just say that you don't know. Don't try to make up an answer.\nALWAYS return a \"SOURCES\" part in your answer.

Documents:
{document}
                                              
Question:
{question}                       

Final answer:
""")
portioned_responses = "\n\n".join(map_results)
prompt = map_prompt.format(
    document=portioned_responses,
    question=query
)

final_answer = llm.invoke(prompt)
print(final_answer)

[llm/start] [llm:OpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: \nUse the following portion of a long document ot see if any of the text is relevant to answer the question.\nReturn any relevant text verbatim.\n\nContext:\nEco Pulse\n\nMC Learn\n\nTrending Topics\n\nSensex Live\n\nLenskart Block Deal\n\nGift Nifty\n\nPetrol and Diesel Prices Today\n\nOnEMI Technology IPO\n\nTata Motors launches Punch iCNG, price starts at Rs 7.1 lakh\n\nThe Punch iCNG is equipped with the company's proprietary twin-cylinder technology with enhanced safety features like a micro-switch to keep the car switched off at the time of refuelling and thermal incident protection that cuts off CNG supply to the engine and releases gas into the atmosphere, Tata Motors said in a statement.\n\nPTI\n\nAugust 04, 2023 / 14:17 IST\n\njoin Us On WhatsApp\n\nFollow Us On Google\n\nAdd as a Preferred Source on Google\n\n\n\nTata Motors launches Punch iCNG, price starts at Rs 7.1 lakh\n\nTEL\n\nWatchlist\

In [5]:
a = ['1','2','34']
b = "\n--\n".join(a)
b

'1\n--\n2\n--\n34'

In [38]:
import pandas as pd

In [41]:
df = pd.read_csv('sample_text.csv')
df.head()

,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps cont...",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event


In [44]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-mpnet-base-v2")

# Remove missing values and convert to strings
texts = df["text"].dropna().astype(str).tolist()

# Generate embeddings
vectors = encoder.encode(texts)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4870.47it/s]


In [45]:
vectors.shape

(8, 768)

In [47]:
import faiss

dim = vectors.shape[1]

# L2 (Euclidean distance) index
index = faiss.IndexFlatL2(dim)

print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7a805865fc90> >


In [48]:
index.add(vectors)

In [49]:
search_query = 'i want to buy a polo t-shirt'
vector  = encoder.encode(search_query)
vector.shape

(768,)

In [52]:
[vector]

[array([ 1.03882849e-02,  2.78686099e-02, -1.18618747e-02,  1.81327239e-02,
         1.21983048e-03, -1.42995743e-02,  1.96229275e-02,  2.07198393e-02,
        -2.23660767e-02,  4.75626737e-02,  1.77976228e-02, -8.00333917e-03,
         2.53419485e-02,  5.26148528e-02,  8.44937935e-03, -1.63944252e-02,
         1.02661615e-02, -2.74856184e-02,  8.37067068e-02, -1.52885346e-02,
         1.67682581e-02, -3.97141092e-03, -2.74348464e-02,  5.02091758e-02,
        -8.36521015e-03, -4.74075824e-02,  2.36915853e-02, -1.01186484e-02,
        -2.82840226e-02,  7.94262160e-03,  4.21435833e-02, -4.19375440e-03,
        -1.91232320e-02, -3.12785432e-02,  1.24685050e-06, -1.04428222e-02,
        -2.19971295e-02, -8.66927952e-02, -1.88496127e-03, -2.54772231e-02,
        -9.72972345e-03,  7.93310702e-02, -3.55963558e-02, -3.05376569e-04,
        -1.12392223e-02, -3.88932452e-02,  5.49313799e-02,  1.35207951e-01,
        -8.19147602e-02,  1.18785053e-02, -9.01844073e-03,  1.92544386e-02,
        -2.8

In [53]:
import numpy as np
svector = np.array(vector).reshape(1, -1)
svector.shape

(1, 768)

In [55]:
vectors

array([[-0.00247394,  0.03626724, -0.0529046 , ..., -0.09152356,
        -0.0397    , -0.04330488],
       [-0.0335727 ,  0.00980516, -0.03250131, ..., -0.05165464,
         0.02245888, -0.03156181],
       [-0.01865323, -0.04051316, -0.01235386, ...,  0.00610587,
        -0.07179647,  0.02773852],
       ...,
       [-0.0006646 ,  0.04252128, -0.05645508, ...,  0.0131547 ,
        -0.0318357 , -0.04357662],
       [-0.03317151,  0.03252458, -0.0248484 , ...,  0.01174421,
         0.05747123,  0.00571023],
       [-0.00166393,  0.00413827, -0.04597083, ...,  0.02008528,
         0.05656241, -0.00161595]], shape=(8, 768), dtype=float32)

In [57]:
distances, I = index.search(svector, k=2)

In [58]:
I

array([[3, 2]])

In [59]:
df.loc[[3,2]]

,text,category
3,Vibrant color jeans for male are becoming a trend,Fashion
2,These are the latest fashion trends for this week,Fashion


In [61]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()
api_key = os.getenv("OPENAI_KEY")

In [63]:
llm  = ChatOpenAI(
    temperature=0.9,
    api_key=api_key
)

In [ ]:
loader = UnstructuredURLLoader(urls = [
    "https://www.moneycontrol.com/news/",
    "https://www.moneycontrol.com/"   

])
data = loader.load()